# 🚦 Traffic Sign Classification (PyTorch) - Training Notebook

Huấn luyện **hai mô hình CNN** phân loại crop biển báo (do YOLOv8 phát hiện), rồi so sánh và ensemble:
- **Custom CNN** (VGG-style, baseline) - `Utils/models.py::CustomCNN`
- **ResNet-18** (tự cài đặt, train từ đầu, chỉnh cho ảnh 64x64) - `Utils/models.py::ResNet18`
- **Ensemble**: trung bình xác suất của hai mô hình

**Input:** ảnh RGB 64x64, giá trị [0, 1]. Model lưu thành `.pt`, copy vào `Inference/`.

**Dataset:** tạo bằng `Data_Pipeline/build_cnn_crops.py` (thư mục `train/val/test/<id>_<tên>/`). Đặt đường dẫn bằng biến môi trường `DATASET_PATH` hoặc sửa ở cell cấu hình.

In [ ]:
import os, sys, copy
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

# Import Utils/models.py (local: ../Utils, Kaggle: models.py cạnh notebook)
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd()))
try:
    from Utils.models import build_model, save_checkpoint
except ImportError:
    from models import build_model, save_checkpoint

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type == 'cpu':
    print("⚠️ Không có GPU. Hãy bật Accelerator trong Session Options!")

## 2. Cấu hình

In [ ]:
DATASET_PATH = os.environ.get('DATASET_PATH', str(Path.cwd().parent.parent / 'Data_Processed' / 'cnn_dataset'))
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR = os.path.join(DATASET_PATH, 'val')
TEST_DIR = os.path.join(DATASET_PATH, 'test')

IMG_SIZE = 64
BATCH_SIZE = 128
EPOCHS = 30
NUM_WORKERS = 4
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

## 3. Dữ liệu & Augmentation
- **Train:** xoay ±10°, zoom nhẹ.
- **Val/Test:** chỉ resize + ToTensor ([0, 1]).

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, scale=(0.9, 1.1)),
    transforms.ToTensor(),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, train_tf)
val_ds = datasets.ImageFolder(VAL_DIR, eval_tf)
test_ds = datasets.ImageFolder(TEST_DIR, eval_tf)

class_names = train_ds.classes
NUM_CLASSES = len(class_names)
print(f"✅ {NUM_CLASSES} lớp: {train_ds.class_to_idx}")
print(f"Train/Val/Test: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

pin = DEVICE.type == 'cuda'
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

# Trọng số lớp (nghịch đảo căn bậc hai tần suất) để giảm lệch dữ liệu
counts = np.bincount(train_ds.targets, minlength=NUM_CLASSES)
class_weights = torch.tensor((counts.sum() / counts) ** 0.5, dtype=torch.float32)
class_weights = (class_weights / class_weights.mean()).to(DEVICE)
print("Số mẫu mỗi lớp:", dict(zip(class_names, counts)))

## 4. Hàm huấn luyện
- Lưu checkpoint theo `val_acc` tốt nhất, `EarlyStopping` theo `val_loss`, `ReduceLROnPlateau`.

In [ ]:
scaler = torch.amp.GradScaler(enabled=DEVICE.type == 'cuda')


def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
                out = model(x)
                loss = criterion(out.float(), y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += y.size(0)
    return total_loss / n, correct / n


def fit(model, kind, ckpt_path, epochs, lr, patience=10):
    criterion = nn.CrossEntropyLoss(weight=class_weights)  # bù lớp ít mẫu
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3, min_lr=1e-7)
    hist = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    best_acc, best_loss, bad = 0.0, float('inf'), 0
    for ep in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader, criterion)
        scheduler.step(vl)
        for k, v in zip(hist, (tl, ta, vl, va)):
            hist[k].append(v)
        print(f"Epoch {ep:02d}/{epochs} | loss {tl:.4f} acc {ta:.4f} | val_loss {vl:.4f} val_acc {va:.4f} | lr {optimizer.param_groups[0]['lr']:.2e}")
        if va > best_acc:
            best_acc = va
            save_checkpoint(model, kind, class_names, ckpt_path)
        if vl < best_loss:
            best_loss, bad = vl, 0
        else:
            bad += 1
            if bad >= patience:
                print("⏹️ Early stopping")
                break
    # Nạp lại checkpoint tốt nhất
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)['state_dict'])
    print(f"✅ Best val_acc: {best_acc:.4f} -> {ckpt_path}")
    return hist

## 5. Custom CNN

In [ ]:
CNN_PATH = 'best_cnn_traffic_sign.pt'
cnn = build_model('cnn', NUM_CLASSES).to(DEVICE)
print(f"Tham số: {sum(p.numel() for p in cnn.parameters()):,}")
history_cnn = fit(cnn, 'cnn', CNN_PATH, EPOCHS, lr=1e-3, patience=10)

## 6. ResNet-18 (tự cài đặt)
Train từ đầu (không pretrained) với LR 1e-3, Adam + giảm LR theo val loss.

In [ ]:
RES_PATH = 'best_resnet18_traffic_sign.pt'
res = build_model('resnet18', NUM_CLASSES).to(DEVICE)
print(f"Tham số: {sum(p.numel() for p in res.parameters()):,}")
history_res = fit(res, 'resnet18', RES_PATH, EPOCHS, lr=1e-3, patience=8)

## 7. Đánh giá & Trực quan hóa

In [ ]:
def plot_history(hist, title):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(hist['accuracy'], label='Train'); plt.plot(hist['val_accuracy'], label='Val')
    plt.title(f'{title} - Accuracy'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(hist['loss'], label='Train'); plt.plot(hist['val_loss'], label='Val')
    plt.title(f'{title} - Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
    plt.show()

def report(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {title}'); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names))

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    return np.concatenate([torch.softmax(model(x.to(DEVICE)), 1).cpu().numpy() for x, _ in loader])

plot_history(history_cnn, 'Custom CNN')
plot_history(history_res, 'ResNet-18')

y_true = np.array(test_ds.targets)  # test_loader không shuffle nên thứ tự khớp
probs = {'Custom CNN': predict_probs(cnn, test_loader),
         'ResNet-18': predict_probs(res, test_loader)}
probs['Ensemble (CNN + ResNet-18)'] = (probs['Custom CNN'] + probs['ResNet-18']) / 2

for name, p in probs.items():
    y_pred = p.argmax(1)
    print(f"\n===== {name} | Test Accuracy: {(y_pred == y_true).mean() * 100:.2f}% =====")
    report(y_true, y_pred, name)

In [ ]:
# Tải model về máy (nếu chạy trên Kaggle)
from IPython.display import FileLink, display
display(FileLink(CNN_PATH))
display(FileLink(RES_PATH))